# Synthetic CDV Experiment - Analysis

This notebook analyzes the multi-seed experiment results across 4 alpha levels.

**Key Questions:**
1. Does CDV matching converge to global performance at alpha=0 (homogeneous)?
2. Does CDV outperform global under heterogeneity (alpha > 0)?
3. How does the advantage scale with alpha?
4. Is the difference statistically significant?

In [ ]:
import osimport json as json_libimport pickleimport numpy as npimport pandas as pdfrom scipy import statsimport matplotlib.pyplot as pltimport seaborn as snsfrom cdv_utils.experiment_runner import load_experiment_resultsfrom cdv_utils.results_analysis import (    prepare_dataframes_for_analysis,    calculate_ate_mse_decomposition,    calculate_cate_rmse)from cdv_utils.visualization import (    plot_scissors_chart,    plot_four_alpha_comparison)plt.style.use("seaborn-v0_8-whitegrid")sns.set_palette("colorblind")# Load configwith open("results/synthetic_experiment_config.json", "r") as f:    config = json_lib.load(f)ALPHA_VALUES = config["alpha_values"]print(f"Alpha values: {ALPHA_VALUES}")print("Analysis loaded and ready.")

## 1. Load All Results

In [ ]:
# Load results for all alpha valuesresults_by_alpha = {}for alpha in ALPHA_VALUES:    path = f"results/synthetic_alpha_{alpha:.2f}.pkl"    if os.path.exists(path):        results_by_alpha[alpha] = load_experiment_results(path)        print(f"alpha={alpha:.2f}: loaded {len(results_by_alpha[alpha])} seeds")    else:        print(f"alpha={alpha:.2f}: NOT FOUND - run experiment first!")print(f"\nLoaded {len(results_by_alpha)} alpha levels.")

## 2. Compute ATE MSE per Alpha

In [ ]:
# Compute ATE-level MSE (bias^2 + variance) for each alphaate_results = {}for alpha, seed_results in results_by_alpha.items():    cdv_ates = []  # per-seed ATE estimates (CDV)    global_ates = []  # per-seed ATE estimates (Global)    true_ates = []  # per-seed ground-truth ATE        for seed_result in seed_results:        if "error" in seed_result:            continue                # CDV ATE: weighted-average of variant-level ATEs        cdv_variant_ates = seed_result.get("cdv_variant_ates", {})        cdv_variant_sizes = seed_result.get("cdv_variant_sizes", {})                if cdv_variant_ates and cdv_variant_sizes:            total_n = sum(cdv_variant_sizes.values())            cdv_ate = sum(                cdv_variant_ates[v] * cdv_variant_sizes[v] / total_n                for v in cdv_variant_ates if v in cdv_variant_sizes            )            cdv_ates.append(cdv_ate)                # Global ATE        global_ate = seed_result.get("global_ate", None)        if global_ate is not None:            global_ates.append(global_ate)                # True ATE        true_ate = seed_result.get("true_ate", None)        if true_ate is not None:            true_ates.append(true_ate)        # Compute MSE decomposition    cdv_ates = np.array(cdv_ates)    global_ates = np.array(global_ates)    true_ates = np.array(true_ates)        if len(cdv_ates) > 0 and len(true_ates) > 0:        cdv_errors = cdv_ates - true_ates[:len(cdv_ates)]        cdv_bias = np.mean(cdv_errors)        cdv_var = np.var(cdv_errors)        cdv_mse = np.mean(cdv_errors**2)                global_errors = global_ates - true_ates[:len(global_ates)]        global_bias = np.mean(global_errors)        global_var = np.var(global_errors)        global_mse = np.mean(global_errors**2)                ate_results[alpha] = {            "cdv_mse": cdv_mse, "cdv_bias": cdv_bias, "cdv_var": cdv_var,            "global_mse": global_mse, "global_bias": global_bias, "global_var": global_var,            "n_seeds": len(cdv_ates)        }                print(f"alpha={alpha:.2f}: CDV MSE={cdv_mse:.4f} (bias={cdv_bias:.4f}, var={cdv_var:.4f}) | "              f"Global MSE={global_mse:.4f} (bias={global_bias:.4f}, var={global_var:.4f})")print("\nATE MSE computation complete.")

## 3. Compute CATE RMSE per Alpha

In [ ]:
# Compute CATE-level RMSE for each alphacate_results = {}for alpha, seed_results in results_by_alpha.items():    cdv_cate_rmses = []    global_cate_rmses = []        for seed_result in seed_results:        if "error" in seed_result:            continue                # CDV CATE RMSE (per-unit error across all variants)        cdv_cate_rmse = seed_result.get("cdv_cate_rmse", None)        if cdv_cate_rmse is not None:            cdv_cate_rmses.append(cdv_cate_rmse)                # Global CATE RMSE        global_cate_rmse = seed_result.get("global_cate_rmse", None)        if global_cate_rmse is not None:            global_cate_rmses.append(global_cate_rmse)        cdv_cate_rmses = np.array(cdv_cate_rmses)    global_cate_rmses = np.array(global_cate_rmses)        if len(cdv_cate_rmses) > 0:        cate_results[alpha] = {            "cdv_mean": np.mean(cdv_cate_rmses),            "cdv_std": np.std(cdv_cate_rmses),            "global_mean": np.mean(global_cate_rmses),            "global_std": np.std(global_cate_rmses),            "n_seeds": len(cdv_cate_rmses)        }                print(f"alpha={alpha:.2f}: CDV RMSE={np.mean(cdv_cate_rmses):.4f} (+/-{np.std(cdv_cate_rmses):.4f}) | "              f"Global RMSE={np.mean(global_cate_rmses):.4f} (+/-{np.std(global_cate_rmses):.4f})")print("\nCATE RMSE computation complete.")

## 4. Statistical Significance Tests

In [ ]:
# Paired t-tests: CDV vs Global at each alpha levelsignificance_results = []for alpha, seed_results in results_by_alpha.items():    cdv_errors_sq = []    global_errors_sq = []        for seed_result in seed_results:        if "error" in seed_result:            continue        cdv_cate_rmse = seed_result.get("cdv_cate_rmse", None)        global_cate_rmse = seed_result.get("global_cate_rmse", None)        if cdv_cate_rmse is not None and global_cate_rmse is not None:            cdv_errors_sq.append(cdv_cate_rmse**2)            global_errors_sq.append(global_cate_rmse**2)        cdv_arr = np.array(cdv_errors_sq)    global_arr = np.array(global_errors_sq)        if len(cdv_arr) > 1:        # Paired t-test: H0: mean(global - cdv) = 0        diff = global_arr - cdv_arr        t_stat, p_value = stats.ttest_rel(global_arr, cdv_arr)        cohens_d = np.mean(diff) / np.std(diff, ddof=1) if np.std(diff, ddof=1) > 0 else 0                significance_results.append({            "alpha": alpha,            "mean_diff": np.mean(diff),            "t_stat": t_stat,            "p_value": p_value,            "cohens_d": cohens_d,            "n_paired": len(cdv_arr),            "significant_05": p_value < 0.05,            "significant_01": p_value < 0.01,            "cdv_better": np.mean(diff) > 0  # positive means global worse        })sig_df = pd.DataFrame(significance_results)print("Statistical Significance (Paired t-test: Global vs CDV on CATE MSE)")print("=" * 80)print(sig_df.to_string(index=False))

## 5. Scissors Chart (MSE vs Alpha)

In [ ]:
# Scissors chart: CDV vs Global MSE as function of alphafig, axes = plt.subplots(1, 2, figsize=(14, 5))# ATE MSE scissorsalphas = sorted(ate_results.keys())cdv_mses = [ate_results[a]["cdv_mse"] for a in alphas]global_mses = [ate_results[a]["global_mse"] for a in alphas]axes[0].plot(alphas, cdv_mses, "o-", label="CDV", color="tab:blue", linewidth=2, markersize=8)axes[0].plot(alphas, global_mses, "s--", label="Global", color="tab:red", linewidth=2, markersize=8)axes[0].set_xlabel("Alpha (Heterogeneity Level)", fontsize=12)axes[0].set_ylabel("ATE MSE", fontsize=12)axes[0].set_title("ATE Estimation Error", fontsize=13)axes[0].legend(fontsize=11)axes[0].set_xticks(alphas)# CATE RMSE scissorsalphas_c = sorted(cate_results.keys())cdv_means = [cate_results[a]["cdv_mean"] for a in alphas_c]global_means = [cate_results[a]["global_mean"] for a in alphas_c]cdv_stds = [cate_results[a]["cdv_std"] for a in alphas_c]global_stds = [cate_results[a]["global_std"] for a in alphas_c]axes[1].errorbar(alphas_c, cdv_means, yerr=cdv_stds, fmt="o-", label="CDV",                 color="tab:blue", linewidth=2, markersize=8, capsize=4)axes[1].errorbar(alphas_c, global_means, yerr=global_stds, fmt="s--", label="Global",                 color="tab:red", linewidth=2, markersize=8, capsize=4)axes[1].set_xlabel("Alpha (Heterogeneity Level)", fontsize=12)axes[1].set_ylabel("CATE RMSE", fontsize=12)axes[1].set_title("CATE Estimation Error", fontsize=13)axes[1].legend(fontsize=11)axes[1].set_xticks(alphas_c)plt.tight_layout()os.makedirs("plots", exist_ok=True)plt.savefig("plots/scissors_chart_synthetic.png", dpi=150, bbox_inches="tight")plt.show()print("Saved: plots/scissors_chart_synthetic.png")

## 6. Bias-Variance Decomposition

In [ ]:
# Bias-Variance decomposition bar chartfig, axes = plt.subplots(1, len(ALPHA_VALUES), figsize=(4*len(ALPHA_VALUES), 5), sharey=True)for i, alpha in enumerate(sorted(ate_results.keys())):    r = ate_results[alpha]        x = [0, 1]    bias2 = [r["cdv_bias"]**2, r["global_bias"]**2]    variance = [r["cdv_var"], r["global_var"]]        axes[i].bar(x, bias2, width=0.4, label="Bias^2", color="tab:orange", alpha=0.8)    axes[i].bar(x, variance, width=0.4, bottom=bias2, label="Variance", color="tab:blue", alpha=0.8)    axes[i].set_xticks(x)    axes[i].set_xticklabels(["CDV", "Global"])    axes[i].set_title(f"alpha = {alpha:.2f}", fontsize=12)    if i == 0:        axes[i].set_ylabel("ATE MSE Decomposition", fontsize=11)    axes[i].legend(fontsize=9)plt.suptitle("Bias-Variance Decomposition across Heterogeneity Levels", fontsize=13, y=1.02)plt.tight_layout()plt.savefig("plots/bias_variance_synthetic.png", dpi=150, bbox_inches="tight")plt.show()print("Saved: plots/bias_variance_synthetic.png")

## 7. Per-Estimator Breakdown

In [ ]:
# Break down results by estimator (if available in results)estimator_results = {}for alpha, seed_results in results_by_alpha.items():    for seed_result in seed_results:        if "error" in seed_result:            continue        est_details = seed_result.get("estimator_details", {})        for est_name, est_data in est_details.items():            if est_name not in estimator_results:                estimator_results[est_name] = {a: {"cdv": [], "global": []} for a in ALPHA_VALUES}            cdv_rmse = est_data.get("cdv_cate_rmse", None)            global_rmse = est_data.get("global_cate_rmse", None)            if cdv_rmse is not None:                estimator_results[est_name][alpha]["cdv"].append(cdv_rmse)            if global_rmse is not None:                estimator_results[est_name][alpha]["global"].append(global_rmse)if estimator_results:    print("Per-Estimator CATE RMSE Summary:")    print("=" * 90)    for est_name, alpha_data in estimator_results.items():        print(f"\n{est_name}:")        for alpha in sorted(alpha_data.keys()):            cdv_vals = alpha_data[alpha]["cdv"]            global_vals = alpha_data[alpha]["global"]            if cdv_vals and global_vals:                print(f"  alpha={alpha:.2f}: CDV={np.mean(cdv_vals):.4f} | Global={np.mean(global_vals):.4f} | "                      f"Diff={np.mean(global_vals)-np.mean(cdv_vals):+.4f}")else:    print("No per-estimator breakdown available in results.")    print("(This requires estimator_details to be saved in seed results)")

## 8. Summary Table for Paper

In [ ]:
# Generate LaTeX-ready summary tablesummary_rows = []for alpha in sorted(ate_results.keys()):    ate_r = ate_results[alpha]    cate_r = cate_results.get(alpha, {})    sig_r = sig_df[sig_df["alpha"] == alpha].iloc[0] if alpha in sig_df["alpha"].values else None        row = {        "Alpha": f"{alpha:.2f}",        "CDV ATE MSE": f"{ate_r['cdv_mse']:.4f}",        "Global ATE MSE": f"{ate_r['global_mse']:.4f}",        "CDV CATE RMSE": f"{cate_r.get('cdv_mean', 0):.4f}",        "Global CATE RMSE": f"{cate_r.get('global_mean', 0):.4f}",        "p-value": f"{sig_r['p_value']:.2e}" if sig_r is not None else "N/A",        "Cohen d": f"{sig_r['cohens_d']:.3f}" if sig_r is not None else "N/A",    }    summary_rows.append(row)summary_df = pd.DataFrame(summary_rows)print("Summary Table for Paper")print("=" * 100)print(summary_df.to_string(index=False))print("\n\nLaTeX format:")print(summary_df.to_latex(index=False))

## 9. Four-Panel Comparison Figure

In [ ]:
# Create the comprehensive 4-panel comparison figurefig, axes = plt.subplots(2, 2, figsize=(14, 10))# Panel 1: ATE MSE scissorsalphas = sorted(ate_results.keys())axes[0,0].plot(alphas, [ate_results[a]["cdv_mse"] for a in alphas],               "o-", label="CDV", color="tab:blue", linewidth=2, markersize=8)axes[0,0].plot(alphas, [ate_results[a]["global_mse"] for a in alphas],               "s--", label="Global", color="tab:red", linewidth=2, markersize=8)axes[0,0].set_xlabel("Alpha")axes[0,0].set_ylabel("ATE MSE")axes[0,0].set_title("(a) ATE MSE vs Heterogeneity")axes[0,0].legend()axes[0,0].set_xticks(alphas)# Panel 2: CATE RMSE scissorsalphas_c = sorted(cate_results.keys())axes[0,1].errorbar(alphas_c, [cate_results[a]["cdv_mean"] for a in alphas_c],                   yerr=[cate_results[a]["cdv_std"] for a in alphas_c],                   fmt="o-", label="CDV", color="tab:blue", linewidth=2, capsize=4)axes[0,1].errorbar(alphas_c, [cate_results[a]["global_mean"] for a in alphas_c],                   yerr=[cate_results[a]["global_std"] for a in alphas_c],                   fmt="s--", label="Global", color="tab:red", linewidth=2, capsize=4)axes[0,1].set_xlabel("Alpha")axes[0,1].set_ylabel("CATE RMSE")axes[0,1].set_title("(b) CATE RMSE vs Heterogeneity")axes[0,1].legend()axes[0,1].set_xticks(alphas_c)# Panel 3: Bias^2 comparisoncdv_bias2 = [ate_results[a]["cdv_bias"]**2 for a in alphas]global_bias2 = [ate_results[a]["global_bias"]**2 for a in alphas]x = np.arange(len(alphas))width = 0.35axes[1,0].bar(x - width/2, cdv_bias2, width, label="CDV", color="tab:blue", alpha=0.8)axes[1,0].bar(x + width/2, global_bias2, width, label="Global", color="tab:red", alpha=0.8)axes[1,0].set_xticks(x)axes[1,0].set_xticklabels([f"{a:.2f}" for a in alphas])axes[1,0].set_xlabel("Alpha")axes[1,0].set_ylabel("Bias^2")axes[1,0].set_title("(c) Bias Squared")axes[1,0].legend()# Panel 4: Effect size (Cohen's d) with significance markersif len(sig_df) > 0:    axes[1,1].bar(range(len(sig_df)), sig_df["cohens_d"].values, color="tab:green", alpha=0.8)    axes[1,1].set_xticks(range(len(sig_df)))    axes[1,1].set_xticklabels([f"{a:.2f}" for a in sig_df["alpha"].values])    axes[1,1].axhline(y=0.2, color="gray", linestyle="--", alpha=0.5, label="Small effect")    axes[1,1].axhline(y=0.5, color="gray", linestyle="-.", alpha=0.5, label="Medium effect")    axes[1,1].axhline(y=0.8, color="gray", linestyle=":", alpha=0.5, label="Large effect")    axes[1,1].set_xlabel("Alpha")    axes[1,1].set_ylabel("Cohen's d")    axes[1,1].set_title("(d) Effect Size (CDV advantage)")    axes[1,1].legend(fontsize=9)plt.tight_layout()plt.savefig("plots/four_panel_synthetic.png", dpi=150, bbox_inches="tight")plt.show()print("Saved: plots/four_panel_synthetic.png")

## 10. Key Findings

**Expected Results:**
- At alpha=0 (no heterogeneity): CDV and Global should produce similar MSE/RMSE
- As alpha increases: CDV should increasingly outperform Global
- The scissors chart should show diverging lines
- Cohen d should reach medium-to-large effect at alpha=1.0

**Interpretation:**
- CDV is safe: it does not harm when heterogeneity is absent
- CDV is beneficial: it captures heterogeneity structure the global model misses
- The advantage grows monotonically with the degree of heterogeneity